# 04 – Evaluation and Statistical Analysis

Loads pre-computed results (from `run_all.py`) and demonstrates the evaluation and statistical comparison pipeline.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_path = Path('../data/results/raw_results.csv')
if not results_path.exists():
    print('Run `python run_all.py` first to generate results.')
else:
    raw_df = pd.read_csv(results_path)
    print(f'Loaded {len(raw_df)} result rows')
    print(raw_df.head())

In [ ]:
# --- Run a mini experiment inline ---
from src.simulation.generator import generate_synthetic_lightcurve
from src.missingness.injector import inject_gaps
from src.imputation.registry import get_imputer
from src.evaluation.metrics import evaluate_imputation
from src.evaluation.aggregator import aggregate_results, summarise_results

t, flux, params = generate_synthetic_lightcurve(N=500, seed=0)
rows = []
for method_key in ['mean_fill', 'forward_fill', 'linear_interp', 'spline_interp']:
    for seed in range(5):
        gapped, missing_idx, true_vals = inject_gaps(flux, p=0.30, seed=seed)
        imp = get_imputer(method_key, seed=seed)
        imputed = imp.impute(t, gapped, missing_idx, period_est=1.0)
        m = evaluate_imputation(t, imputed, missing_idx, true_vals,
                                true_period=1.0, true_amplitude=0.1, true_phase=0.0)
        m.update({'method': imp.name, 'method_key': method_key, 'fraction': 0.30, 'seed': seed})
        rows.append(m)

mini_df = aggregate_results(rows)
summary = summarise_results(mini_df)
print(summary[['method', 'fraction', 'rmse_mean', 'rmse_std', 'prr']].to_string(index=False))

In [ ]:
# Statistical tests
from src.statistics.tests import wilcoxon_signed_rank, friedman_test, bootstrap_ci

methods = mini_df['method'].unique().tolist()
rmse_matrix = np.column_stack([mini_df[mini_df['method'] == m]['rmse'].values for m in methods])

stat, p = friedman_test(rmse_matrix)
print(f'Friedman test: statistic={stat:.3f}, p={p:.4f}')

for m in methods[1:]:
    x = mini_df[mini_df['method'] == methods[0]]['rmse'].values
    y = mini_df[mini_df['method'] == m]['rmse'].values
    _, p_w = wilcoxon_signed_rank(x, y)
    lo, hi = bootstrap_ci(y)
    print(f'  {methods[0]} vs {m}: Wilcoxon p={p_w:.4f}  95% CI=[{lo:.4f}, {hi:.4f}]')